# Hello VLA

Welcome to your very first hands on lesson. In this notebook you will:

1. Confirm your Colab runtime is working.
2. Load a real multimodal model (SmolVLM) and ask it to describe an image.
3. Download a real Vision-Language-Action model (SmolVLA) and look inside it.

Run each cell in order by pressing **Shift + Enter**.

## Step 1. Turn on a free GPU (optional)

Click **Runtime** then **Change runtime type**, pick **T4 GPU**, then **Save**. If no GPU is available, CPU still works, just slower.

In [ ]:
import warnings, logging, os
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)

import torch

device = 0 if torch.cuda.is_available() else -1
print('Using device:', 'cuda' if device == 0 else 'cpu')
print('PyTorch version:', torch.__version__)

## Step 2. Ask SmolVLM to describe an image

A VLA is made of two halves: a **vision language model** that understands images and text, and an **action expert** that turns that understanding into robot actions. We are going to start by running the vision language half on its own, using Hugging Face's high level `pipeline` API which handles all the loading details for us.

In [ ]:
!pip install -q -U transformers

In [ ]:
from transformers import pipeline

model_id = 'HuggingFaceTB/SmolVLM-500M-Instruct'
dtype = torch.bfloat16 if device == 0 else torch.float32

vlm = pipeline(
    task='image-text-to-text',
    model=model_id,
    dtype=dtype,
    device=device,
)

vlm.model.generation_config.max_length = None
vlm.model.generation_config.max_new_tokens = 120

print('Loaded', model_id)

In [ ]:
from PIL import Image, ImageDraw

image = Image.new('RGB', (384, 256), color=(200, 220, 240))
draw = ImageDraw.Draw(image)
draw.rectangle([60, 140, 160, 220], fill=(220, 40, 40), outline=(0, 0, 0), width=3)
draw.ellipse([220, 140, 320, 220], fill=(40, 160, 60), outline=(0, 0, 0), width=3)
draw.rectangle([0, 220, 384, 256], fill=(120, 90, 50))
image

In [ ]:
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': 'Describe this scene in one sentence, then list every shape and color you can see.'},
        ],
    }
]

result = vlm(text=messages)
print(result[0]['generated_text'][-1]['content'])

Stop and notice what just happened. You gave a neural network an image plus a question in English, and it answered in English. This same kind of model is the brain of every modern VLA. The only piece missing is a small network that turns the understanding into joint angles. That piece is called the **action expert**, and together they form a Vision-Language-Action model.

## Step 3. Look inside a real VLA

Now we are going to download the actual **SmolVLA** model from Hugging Face and look at what is inside it. Instead of using a high level wrapper library, we will download the model files directly from the Hugging Face Hub and read them ourselves with `safetensors`. This is more transparent and shows you exactly what is inside a 450 million parameter Vision-Language-Action model.

In [ ]:
!pip install -q huggingface_hub safetensors

In [ ]:
from huggingface_hub import snapshot_download
from safetensors import safe_open
import json

print('Downloading SmolVLA from lerobot/smolvla_base ...')
local_dir = snapshot_download(
    repo_id='lerobot/smolvla_base',
    allow_patterns=['*.safetensors', '*.json'],
)
print('Downloaded to:', local_dir)
print()

config_path = os.path.join(local_dir, 'config.json')
if os.path.exists(config_path):
    with open(config_path) as f:
        config = json.load(f)
    print('Model type:           ', config.get('type', 'smolvla'))
    print('Chunk size (actions): ', config.get('chunk_size', 'unknown'))
    print('Action dimension:     ', config.get('max_action_dim', 'unknown'))
    print()

shards = [f for f in os.listdir(local_dir) if f.endswith('.safetensors')]
total_params = 0
top_modules = {}

for shard in shards:
    with safe_open(os.path.join(local_dir, shard), framework='pt') as f:
        for key in f.keys():
            tensor = f.get_tensor(key)
            total_params += tensor.numel()
            top = key.split('.')[0]
            top_modules[top] = top_modules.get(top, 0) + tensor.numel()

print(f'Total parameters: {total_params/1e6:.1f}M')
print()
print('Top level modules inside SmolVLA:')
for name, n in sorted(top_modules.items(), key=lambda x: -x[1]):
    print(f'  {name:30s} {n/1e6:7.1f}M parameters')

## What you just did

1. You ran a 500M parameter multimodal model on an image and got an English answer.
2. You downloaded a real 450M parameter Vision-Language-Action model and saw, broken down by component, exactly what is inside it.
3. You did all of this in a browser, for free, on a machine you do not own.

You are now standing at the finish line of the course. In the next 19 modules we walk back to the start and build up every piece of what you just ran, from NumPy and gradient descent, to the Transformer, to modern robotics math, all the way back to fine tuning a VLA on a real robot dataset.

See you in Module 1.